In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw06.ipynb")

# HW 06 — Permutation Testing and Missing Values

Investigate factory differences in Skittles and missing credit-card information in payment records. Complete the short coding questions, visualizations, and written interpretation.

**Grading:** 75 autograded points and 25 manually graded points.

Submit your notebook to **Gradescope by the deadline listed in Canvas**. Restart the kernel, run all cells, and save your notebook before submitting. Retain the plot outputs and written response for manual grading. Submit early to allow time for troubleshooting and support.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

plt.rcParams['figure.figsize'] = (8, 5)
colors = ['green', 'orange', 'purple', 'red', 'yellow']

### Required Datasets

- `data/skittles.tsv`: Bag-level color counts and the `Factory` column (`Yorkville` or `Waco`).
- `data/payment.csv`: Payment records with `date_of_birth` and `credit_card_number`.

Use the original data files. Do not modify input DataFrames inside your functions. Random simulations use the supplied seeds so results are reproducible.

In [ ]:
skittles = pd.read_csv(Path('data') / 'skittles.tsv', sep='	')
payments = pd.read_csv(Path('data') / 'payment.csv')
skittles.head()

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Part 1: Permutation Testing

Under the null hypothesis, bags from the two factories have the same distribution. Permutation tests randomly reassign factory labels while preserving the group sizes. Use absolute mean differences for one color and total variation distance (TVD) for all five colors.

<br>

---
## Question 1 — 10 points

Complete `diff_of_means(df, col='orange')` to:

- Find the average number of candies per bag for the specified color at each factory: **Yorkville** and **Waco**.
- Return the **absolute difference** between these two averages as a **float**.

Use the color specified by `col`, which defaults to `'orange'`.

Hint: Use groupby or Boolean selection.

In [ ]:

def diff_of_means(df, col='orange'):
    means = ...
    difference = ...
    return difference

In [ ]:
grader.check("q1")

### Question 2 — 15 points

Complete `color_test(df, col='orange', n_reps=1000, seed=42)` to perform a permutation test for the requested color.

- `simulated`: An array of simulated absolute mean differences.
- `pvalue`: The proportion of simulated statistics at least as large as the observed statistic.

Return `(pvalue, simulated)`. Complete the blanks in the supplied loop.

*Hint:* Reassign the shuffled values to the existing factory groups.

In [ ]:

def color_test(df, col='orange', n_reps=1000, seed=42):
    rng = np.random.default_rng(seed)
    values = df[col].to_numpy()
    yorkville = df['Factory'].eq('Yorkville').to_numpy()
    observed = diff_of_means(df, col)
    simulated = np.empty(n_reps)
    for i in range(n_reps):
        shuffled = rng.permutation(values)
        simulated[i] = ...
    pvalue = ...
    return (pvalue, simulated)

In [ ]:
grader.check("q2")

In [ ]:
# Provided results for Questions 3 and 4.
orange_observed = diff_of_means(skittles)
orange_p, orange_simulated = color_test(skittles)
print('Orange p-value:', orange_p)

## Question 3 — 13 points

Complete `factory_tvd(df)` to calculate the **total variation distance (TVD)** between the candy color distributions at **Yorkville** and **Waco**.

- `totals`: Total candy counts for each color at each factory, using `colors`.
- `proportions`: Color proportions within each factory.
- `tvd`: TVD between the two factories, returned as a **float**.

*Hint:* TVD measures how different two categorical distributions are.

In [ ]:

def factory_tvd(df):
    totals = ...
    proportions = ...
    tvd = ...
    return tvd

In [ ]:
grader.check("q3")

In [ ]:
# Provided TVD permutation test: run without editing.
tvd_observed = factory_tvd(skittles)
tvd_rng = np.random.default_rng(42)
tvd_simulated = np.empty(1000)
for i in range(1000):
    permuted = skittles.assign(Factory=tvd_rng.permutation(skittles['Factory'].to_numpy()))
    tvd_simulated[i] = factory_tvd(permuted)
tvd_p = float(np.mean(tvd_simulated >= tvd_observed))
print('TVD statistic:', tvd_observed, '| p-value:', tvd_p)

<!-- BEGIN QUESTION -->

### Question 4 — 12 points (Manual)

Create **two side-by-side histograms** of `orange_simulated` and `tvd_simulated`.

- Mark the corresponding observed statistic with a vertical line in each plot.
- Include descriptive titles, axis labels, and legends.
- Display both plots in the notebook.

This question is manually graded: histograms 4 points, observed-statistic markers 4 points, labels and legends 4 points.

In [ ]:

# Create and display your two plots here.
...

<!-- END QUESTION -->

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Part 2: Missingness

- **MD:** Values are deliberately not collected under the collection rules.
- **MCAR:** Missingness is unrelated to observed and missing values.
- **MAR:** After accounting for observed variables, missingness does not depend on the missing value itself.
- **NMAR (MNAR):** Missingness still depends on the missing value after accounting for observed variables.

Associations with observed variables can provide evidence against MCAR, but cannot establish MAR versus NMAR.

### Question 5 — 10 points

Assign each scenario its missingness mechanism: `'MD'`, `'MCAR'`, `'MAR'`, or `'NMAR'`. Store the answers in order in `missingness_answers`.

1. Delivery addresses are not collected for pickup orders, as specified by the ordering system.
2. A random software failure deletes phone numbers independently of customer information.
3. Response to an income question depends on recorded employment status; within each employment category, it is unrelated to income.
4. Customers with lower satisfaction are less likely to answer the satisfaction question, even after accounting for recorded information.
5. Previous-sport details are intentionally skipped when a student reports playing no sports.

In [ ]:

missingness_answers = ...

In [ ]:
grader.check("q5")

### Payment Records

The original payment records describe purchases on January 1, 2019. For consistency with the original homework, use **2024 as a fixed reference year** and calculate approximate age as `2024 - birth year`; this is not age at purchase. Exclude rows with missing or unparseable birth dates from the age comparisons. Do not fill in missing ages.

### Question 6 — 12 points

Complete `prepare_payments(df)` to create a copy containing:

- `cust_age`: Approximate age using the fixed reference year 2024.
- `missing_cc`: A Boolean indicator for missing `credit_card_number`.

Remove rows without a valid `cust_age` and return the resulting DataFrame. Preserve the original row indices.

*Hint:* Use `pd.to_datetime` with `errors='coerce'` and `dropna`.

In [ ]:

def prepare_payments(df):
    result = df.copy()
    result['cust_age'] = ...
    result['missing_cc'] = ...
    return ...

In [ ]:
grader.check("q6")

In [ ]:
# Provided age groups: run without editing.
with_age = prepare_payments(payments)
age_missing = with_age.loc[with_age['missing_cc'], 'cust_age'].to_numpy()
age_present = with_age.loc[~with_age['missing_cc'], 'cust_age'].to_numpy()
if len(age_missing) == 0 or len(age_present) == 0:
    raise ValueError('Age comparisons require observations in both missingness groups.')

# Provided permutation test using the absolute mean difference.
age_rng = np.random.default_rng(42)
age_observed = float(abs(age_missing.mean() - age_present.mean()))
age_pooled = np.concatenate([age_missing, age_present])
age_simulated = np.empty(1000)
for i in range(1000):
    shuffled = age_rng.permutation(age_pooled)
    age_simulated[i] = abs(shuffled[:len(age_missing)].mean() - shuffled[len(age_missing):].mean())
age_mean_p = float(np.mean(age_simulated >= age_observed))
print('Age mean-difference p-value:', age_mean_p)

### Question 7 — 15 points

Run a **two-sample K–S test** comparing `age_missing` and `age_present` using `method='auto'`.

- `age_ks_stat`: The K–S statistic.
- `age_ks_p`: The K–S p-value.
- `age_ks_reject`: Whether to reject at the 0.05 significance level (`True` or `False`).

The K–S test compares entire distributions; it is different from the supplied permutation test based on means.

*Hint:* The result has `.statistic` and `.pvalue` attributes.

In [ ]:

age_result = ...
age_ks_stat = ...
age_ks_p = ...
age_ks_reject = ...

In [ ]:
grader.check("q7")

<!-- BEGIN QUESTION -->

### Question 8 — 8 points (Manual)

Create an **overlaid density histogram** comparing `age_missing` and `age_present`. Use the same bin boundaries for both groups so the distributions can be compared fairly.

Include a descriptive title, labeled axes, a legend identifying credit-card missingness, and transparency so both distributions are visible.

This question is manually graded: comparable density histograms 4 points; title, labels, legend, and readability 4 points.

In [ ]:

# Create and display your age-distribution plot here.
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Question 9 — 5 points (Manual)

Write **3–5 sentences** comparing the age mean-difference permutation test and the age K–S test.

- Report both p-values and the corresponding decisions at 0.05.
- Explain why the tests may lead to different conclusions, referring to your Q8 plot.
- State what these results can and cannot establish about the missingness mechanism.

Manual grading: p-values and decisions 2 points; comparison of statistics and plot 2 points; limits of the missingness conclusion 1 point.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Congratulations! You have finished HW06! ##

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers.  If you run into any issues when running this cell, feel free to check the [Debugging Guide](https://mtu.instructure.com/courses/1619791/pages/debugging-guide).

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)